# Camada Bronze — Ingestão Batch

Este notebook realiza a leitura e validação dos arquivos brutos armazenados na camada Bronze do Amazon S3.

Os arquivos são preservados em formato CSV, sem aplicação de regras de negócio ou transformações estruturais nesta etapa.

Objetivos:
- validar a disponibilidade dos arquivos;
- realizar a leitura com Spark;
- validar quantidade de registros, colunas e schema;
- registrar metadados básicos da ingestão;
- gerar evidências da execução da camada Bronze.

In [0]:
# Define o caminho da camada Bronze no Amazon S3 utilizado pela pipeline.

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"
BRONZE_PATH = f"s3://{BUCKET_NAME}/bronze/"

print(f"Bronze path: {BRONZE_PATH}")

In [0]:
# Lista os arquivos disponíveis na camada Bronze para validar a existência e o acesso às fontes de dados.

arquivos_bronze = dbutils.fs.ls(BRONZE_PATH)
print(f"Quantidade de arquivos encontrados: {len(arquivos_bronze)}\n")

for arquivo in arquivos_bronze:
    print(
        f"Nome: {arquivo.name} | "
        f"Tamanho: {arquivo.size} bytes"
    )

## Leitura dos datasets

Os arquivos da camada Bronze são carregados utilizando Apache Spark.

In [0]:
# Mapeia os datasets esperados para seus respectivos arquivos CSV armazenados na camada Bronze.

DATASETS = {
    "avaliacao_alfabetizacao_municipio": f"{BRONZE_PATH}avaliacao_alfabetizacao_municipio.csv",
    "avaliacao_alfabetizacao_uf": f"{BRONZE_PATH}avaliacao_alfabetizacao_uf.csv",
    "meta_alfabetizacao_brasil": f"{BRONZE_PATH}meta_alfabetizacao_brasil.csv",
    "meta_alfabetizacao_municipio": f"{BRONZE_PATH}meta_alfabetizacao_municipio.csv",
    "meta_alfabetizacao_uf": f"{BRONZE_PATH}meta_alfabetizacao_uf.csv",
    "avaliacao_alunos": f"{BRONZE_PATH}avaliacao_alunos.csv"
}

print(f"Datasets configurados: {len(DATASETS)}")

for nome, caminho in DATASETS.items():
    print(f"{nome}: {caminho}")

In [0]:
# Realiza a leitura dos arquivos CSV com Spark e armazena os DataFrames carregados para as validações da Bronze.

dataframes_bronze = {}

for nome, caminho in DATASETS.items():
    try:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(caminho)
        )

        dataframes_bronze[nome] = df

        print(f"[OK] {nome}")

    except Exception as erro:
        print(f"[ERRO] {nome}: {erro}")

In [0]:
# Valida a quantidade de registros, número de colunas e schema inferido pelo Spark para cada dataset.

resumo_bronze = []

for nome, df in dataframes_bronze.items():
    qtd_registros = df.count()
    qtd_colunas = len(df.columns)

    resumo_bronze.append({
        "dataset": nome,
        "registros": qtd_registros,
        "colunas": qtd_colunas
    })

    print("=" * 80)
    print(f"Dataset: {nome}")
    print(f"Registros: {qtd_registros}")
    print(f"Colunas: {qtd_colunas}")
    print("Schema:")
    df.printSchema()

In [0]:
# Consolida as métricas dos datasets em uma tabela única para facilitar a validação e gerar evidências da ingestão.

from pyspark.sql import Row

df_resumo_bronze = spark.createDataFrame(
    [Row(**item) for item in resumo_bronze]
)

display(df_resumo_bronze.orderBy("dataset"))

In [0]:
# Registra metadados da execução da Bronze, incluindo camada, formato, status e horário do processamento.

from datetime import datetime, timezone

data_hora_processamento = datetime.now(timezone.utc)
metadados_bronze = []

for item in resumo_bronze:
    metadados_bronze.append({
        **item,
        "camada": "bronze",
        "formato_origem": "csv",
        "status": "SUCESSO",
        "data_hora_processamento_utc": data_hora_processamento
    })

df_metadados_bronze = spark.createDataFrame(metadados_bronze)
display(df_metadados_bronze.orderBy("dataset"))

In [0]:
# Verifica se todos os datasets obrigatórios foram carregados e interrompe a execução caso alguma fonte esteja ausente.

datasets_esperados = set(DATASETS.keys())
datasets_carregados = set(dataframes_bronze.keys())
faltantes = datasets_esperados - datasets_carregados

if faltantes:
    raise Exception(
        f"Falha na ingestão Bronze. Datasets não carregados: {faltantes}"
    )

print(
    f"[OK] Validação concluída: "
    f"{len(datasets_carregados)}/{len(datasets_esperados)} datasets carregados."
)

## Resultado da ingestão Bronze

A execução da camada Bronze confirmou:

- disponibilidade das 6 fontes de dados no Amazon S3;
- leitura dos arquivos CSV utilizando Apache Spark;
- validação da quantidade de registros e colunas;
- inspeção dos schemas inferidos;
- registro de metadados da execução;
- tratamento de falhas durante a leitura;
- validação da presença de todos os datasets esperados.

Os arquivos originais permanecem inalterados em formato CSV na camada Bronze. Transformações, padronizações e regras de qualidade serão aplicadas posteriormente na camada Silver.